In [0]:
#Import libraries
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType, IntegerType
from pyspark.sql.functions import trim,col

In [0]:
# Read data bronze table and load to dataframe
df = spark.table("baraa_projectwork.bronze.crm_cust_info")


### Transformation

In [0]:
# Remove extra spaces from all string columns
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
    df = df.withColumn(field.name, trim(col(field.name)))

### Normalization

In [0]:
# Normalize marital status and gender 
df = (
    df.withColumn(
        "cst_marital_status",
        F.when(F.upper(F.col("cst_marital_status")) == "S", "Single")
         .when(F.upper(F.col("cst_marital_status")) == "M", "Married")
         .otherwise("n/a")
    ).withColumn(
        "cst_gndr",
        F.when(F.upper(F.col("cst_gndr")) == "F", "Female")
         .when(F.upper(F.col("cst_gndr")) == "M", "Male")
         .otherwise("n/a")
    )
)

### Clear away records without Customer ID

In [0]:
# current record count
row_count = df.count()
display(spark.createDataFrame([(row_count,)], ["row_count"]))

# remove records with missing customerid
df = df.filter(col("cst_id").isNotNull())

# new record count after records with missing customerid
row_count = df.count()
display(spark.createDataFrame([(row_count,)], ["row_count"]))

### Rename Columns

In [0]:
RENAME_MAP = [
  ("cst_id", "customer_id"),
  ("cst_key", "customer_number"),
  ("cst_firstname", "first_name"),
  ("cst_lastname", "last_name"),
  ("cst_marital_status", "marital_status"),
  ("cst_gndr", "gender"),
  ("cst_create_date", "created_date")
]

for old_name, new_name in RENAME_MAP:
  df = df.withColumnRenamed(old_name, new_name)

In [0]:
df.limit(10).display()


### Writing to Silver table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("baraa_projectwork.silver.crm_customers")

In [0]:
%sql
SELECT * FROM baraa_projectwork.silver.crm_customers LIMIT 10